# 03 — Leakage-safe ML dataset construction

| Item | Definition |
|---|---|
| **Scientific purpose** | Convert complete Stage-02 realizations into normalized, traceable training/validation/test tensors without realization leakage. |
| **Inputs** | Stage-02 exact-physics realizations and manifest. |
| **Outputs** | Immutable realization files with low-frequency priors, realization split IDs, training-only normalization, multiscale patch index, integrity report, and dataset manifest. |
| **Data availability** | Schemas and algorithms are public; generated realization arrays remain local. |
| **Local/private-data requirements** | Ignored `configs/paths.yaml` and a completed or explicitly subset-labeled Stage-02 artifact directory. No toy fallback is used. |
| **Software requirements** | `pip install -e ".[ml,notebooks]"`. |
| **Approximate runtime** | Minutes for indexing and prior construction; storage and time scale with realization count. |
| **Pipeline position** | Consumes Notebook 02; supplies the immutable tensors used by Notebooks 04 and 05. |

This experiment is **AVO-guided refinement of a supplied low-frequency elastic prior**. It is not unconstrained AVO-only absolute-property inversion.

In [ ]:
from pathlib import Path

def find_repository_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "sage_avo").exists():
            return candidate
    raise RuntimeError("Run this notebook from the installed SAGE-AVO repository.")

ROOT = find_repository_root()

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sage_avo.config import load_config, seed_everything
from sage_avo.data import IndexedRealizationPatches
from sage_avo.experiments import build_stage03_dataset, validate_dataset_integrity

workflow = load_config(ROOT / "configs" / "ml_dataset_s01.yaml")
paths_file = ROOT / "configs" / "paths.yaml"
if not paths_file.exists():
    raise FileNotFoundError("Create ignored configs/paths.yaml from configs/paths.example.yaml.")
paths = load_config(paths_file)
seed_everything(int(workflow["stage"]["seed"]))

private_root = Path(paths["private_artifact_root"])
stage02_dir = private_root / "stage_artifacts" / "stage02" / "realizations"
dataset_dir = private_root / "stage_artifacts" / "stage03" / "dataset"
figure_dir = private_root / "figures" / "stage03"
figure_dir.mkdir(parents=True, exist_ok=True)

## 1. Validate the realization contract

Required channels are near/mid/far AVO, Vp/Vs/density targets, RGT, segmentation, DELTA, P(sand), porosity, plume mask, and valid mask. Shapes and finite values are checked before any split or patch is produced. Realization IDs—not patches—are the independent sampling units.

In [ ]:
source_manifest_path = stage02_dir / "manifest.json"
if not source_manifest_path.exists():
    raise FileNotFoundError("Run Notebook 02 against the licensed/generated inputs first.")
source_manifest = json.loads(source_manifest_path.read_text())
display(pd.Series({
    "source_status": source_manifest["status"],
    "realizations": source_manifest["generated_realizations"],
    "exact_forward_operator": source_manifest["exact_forward_operator"],
    "delta_convention": source_manifest["delta_convention"],
}).to_frame("value"))

## 2. Split before patching

A seeded permutation assigns entire realizations to train/validation/test (70/15/15 in the production configuration). Patch coordinates are generated only after this assignment. Consequently, no geological deformation, plume scenario, or trace from one realization can leak across splits.

## 3. Disclosed low-frequency prior

The synthetic prior is derived from each target/truth elastic cube using a Gaussian approximation to a 2 Hz low-pass filter. With `dt = 0.004 s` and `sigma_constant = 0.133`,

\[
\sigma_t=\frac{0.133}{f_c\,\Delta t}=16.625\ \text{samples},
\qquad \sigma_x=2\sigma_t=33.25\ \text{traces}.
\]

These exact parameters and the boundary mode are saved in `dataset_manifest.json`. Normalization is fitted from full **training realizations only**, then applied unchanged to validation and test.

In [ ]:
prior = workflow["prior"]
sigma_time = prior["sigma_constant"] / (prior["cutoff_hz"] * prior["dt_seconds"])
sigma_trace = sigma_time * prior["lateral_sigma_ratio"]
display(pd.Series({**prior, "sigma_time_samples": sigma_time, "sigma_trace_samples": sigma_trace}).to_frame("value"))

## 4. Build the immutable dataset

Raw patches are sampled at 40×80, 50×100, and 64×128 physical extents with configured proportions, then resized to 50×100 tensors. Continuous channels use bilinear interpolation; class labels and masks use nearest-neighbor interpolation. Every row preserves realization ID, origin, raw size, tensor size, and both resize factors.

In [ ]:
manifest = build_stage03_dataset(
    config=workflow,
    paths=paths,
    source_directory=stage02_dir,
    output_directory=dataset_dir,
)
integrity = validate_dataset_integrity(dataset_dir)
display(pd.Series(integrity).to_frame("value"))

## 5. Split, normalization, and patch metadata audit

In [ ]:
split_ids = json.loads((dataset_dir / "split_ids.json").read_text())
normalization = json.loads((dataset_dir / "normalization.json").read_text())
patch_index = pd.read_csv(dataset_dir / "patch_index.csv")

display(pd.DataFrame({name: pd.Series(values) for name, values in split_ids.items()}))
display(pd.DataFrame(normalization, index=["near/Vp", "mid/Vs", "far/density"]))
display(patch_index.groupby(["split", "raw_height", "raw_width"]).size().rename("patches").to_frame())
display(patch_index.head())

sets = {name: set(values) for name, values in split_ids.items()}
assert sets["train"].isdisjoint(sets["validation"])
assert sets["train"].isdisjoint(sets["test"])
assert sets["validation"].isdisjoint(sets["test"])

## 6. Tensor contract and representative patches

For class-channel QC, one patch is selected in each split by a reproducible stratified rule: seeded random selection among patches containing sand or plume. This prevents an all-background panel while remaining independent of model performance. AVO and elastic values are normalized using the saved training statistics in the loader; RGT and categorical targets retain their native meanings. Metadata enables predictions to be traced back to the full realization.

In [ ]:
datasets = {split: IndexedRealizationPatches(dataset_dir, split) for split in ("train", "validation", "test")}
rng = np.random.default_rng(int(workflow["stage"]["seed"]))
samples = {}
selected_indices = {}
for split, dataset in datasets.items():
    candidates = [index for index in range(len(dataset)) if (dataset[index]["segmentation"] > 0).any()]
    if not candidates:
        raise ValueError(f"No sand/plume patch is available for {split} QC")
    selected_indices[split] = int(rng.choice(candidates))
    samples[split] = dataset[selected_indices[split]]
print({split: len(dataset) for split, dataset in datasets.items()})
print("Seeded class-QC patch indices:", selected_indices)
print({key: tuple(value.shape) for key, value in samples["train"].items() if hasattr(value, "shape")})

columns = [
    ("avo", 0, "Near AVO"), ("avo", 1, "Mid AVO"), ("avo", 2, "Far AVO"),
    ("low", 0, "Low-frequency Vp"), ("target", 0, "Target Vp"),
    ("target", 1, "Target Vs"), ("target", 2, "Target density"),
    ("segmentation", None, "Facies/plume"), ("rgt", None, "RGT"),
]
fig, axes = plt.subplots(3, len(columns), figsize=(20, 7), constrained_layout=True)
for row, split in enumerate(("train", "validation", "test")):
    sample = samples[split]
    for col, (key, channel, title) in enumerate(columns):
        values = sample[key].numpy()
        panel = values[channel] if channel is not None else values
        cmap = "gray" if key == "avo" else ("tab10" if key == "segmentation" else "viridis")
        axes[row, col].imshow(panel, aspect="auto", cmap=cmap)
        axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
        if row == 0: axes[row, col].set_title(title)
        if col == 0: axes[row, col].set_ylabel(split)
qc_path = figure_dir / "stage03_split_patch_contract.png"
fig.savefig(qc_path, dpi=300, bbox_inches="tight")
plt.show()
print("private figure:", qc_path)

## 7. Distribution QC without patch-pooling claims

In [ ]:
rows = []
for split, dataset in datasets.items():
    for index in np.linspace(0, len(dataset) - 1, min(40, len(dataset)), dtype=int):
        sample = dataset[int(index)]
        for channel, name in enumerate(("Vp", "Vs", "density")):
            rows.append({
                "split": split,
                "property": name,
                "normalized_mean": float(sample["target"][channel].mean()),
                "normalized_std": float(sample["target"][channel].std()),
            })
distribution = pd.DataFrame(rows)
display(distribution.groupby(["split", "property"])[["normalized_mean", "normalized_std"]].agg(["mean", "std"]))

## Stage outputs

| artifact | shape/type | scientific meaning | consumed by |
|---|---|---|---|
| `realizations/*.npz` | full images with AVO, truth, prior, RGT, masks/classes | Immutable whole-realization evaluation unit | Notebooks 04–05 |
| `split_ids.json` | realization-ID lists | Leakage-safe partition | Notebooks 04–05 |
| `normalization.json` | 3-channel means/stds | Training-only normalization transform | Notebooks 04–05 |
| `patch_index.csv` | one row per multiscale patch | Traceable sampling and resize metadata | Notebook 04 |
| `dataset_manifest.json` | prior, channels, split, integrity | Complete ML task contract | Notebooks 04–05 |

## Scientific checks

- Required channels, matching dimensions, and finite values are validated before splitting.
- Train/validation/test realization-ID sets are asserted disjoint.
- Patch rows are checked against their assigned realization split.
- Low-frequency priors are explicitly labeled truth-derived and their smoothing constants are saved.
- Normalization statistics use training realizations only.
- Continuous and categorical resize modes are separated, while raw physical sizes and scale factors remain in metadata.

## Next stage

Notebook 04 consumes the normalized near/mid/far AVO, truth-derived low-frequency Vp/Vs/density prior, RGT, valid masks, and elastic/segmentation targets. It trains controlled SAGE-AVO variants against the same immutable split and checkpoint rule.